# MNIST MLP

A multilayer perceptron in PyTorch that classifies handwritten digits.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## 1. Load the data

- `transforms.ToTensor()` converts images from PIL format to PyTorch tensors and scales pixels from 0-255 to 0-1
- `transforms.Normalize((0.1307,), (0.3081,))` normalises to mean 0, std 1 using MNIST's precomputed stats
- `DataLoader` wraps the dataset to give batches of (images, labels) on each iteration


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Sanity check — what does one batch look like?
images, labels = next(iter(train_loader))
print(f'Batch of images shape: {images.shape}')
print(f'Batch of labels shape: {labels.shape}')
print(f'Single image shape: {images[0].shape}')
print(f'Pixel value range: [{images.min():.2f}, {images.max():.2f}]')

## 2. Define the model

- Images arrive as (batch, 1, 28, 28) and are flattened to (batch, 784) with `x.view(x.size(0), -1)`
- `nn.CrossEntropyLoss` applies softmax internally, so the output layer produces raw logits


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 100)
        self.layer2 = nn.Linear(100, 100)
        self.layer3 = nn.Linear(100, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.tanh(self.layer1(x))
        x = torch.tanh(self.layer2(x))
        x = self.layer3(x)
        return x

## 3. Set up training


In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

## 4. Training loop


In [ ]:
num_epochs = 5
losses = []
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        # zero the gradients before each step
        optimizer.zero_grad()
        # forward pass, loss, backward, update
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()

## 5. Plot the loss curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Training step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

## 6. Evaluate on test set

- `model.eval()` switches off dropout and batchnorm training behaviour
- `torch.no_grad()` disables gradient tracking
- The predicted class is the argmax of the logits


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test accuracy: {accuracy:.2f}%')

if accuracy > 95:
    print('✅ Target met: >95% accuracy')
else:
    print(f'❌ Below target. Need to improve by {95 - accuracy:.2f}%')